In [2]:
# =============================================================================
# NOTEBOOK: materialize_view.ipynb
#
# DESCRIPCIÓN:
#   Notebook genérico para materializar una vista de forma incremental en una
#   tabla física dentro de la capa Silver. Es controlado por metadatos.
#
# PARÁMETROS:
#   - task_id (integer): El ID de la tarea a ejecutar.
# =============================================================================

# --- Importaciones ---
from pyspark.sql.functions import row_number, col, lit, max as spark_max, expr
from pyspark.sql import functions as F, types as T
from datetime import datetime
from pyspark.sql.window import Window
import notebookutils
import time
import random

# Configuración para LEER fechas antiguas de fuentes Parquet/Delta
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")

# Configuración para ESCRIBIR fechas antiguas a destinos Parquet/Delta (RECOMENDADA)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")


# --- Parámetros ---
# Este notebook usará un task_id para identificar la fila de forma única.
task_id = 51

# --- Configuración Global ---
# Apunta al atajo o nombre completo de la tabla de control
control_table_full_name = "lh_control_erp.dbo.bronze_to_silver_control" 
current_utc_timestamp = datetime.utcnow()

StatementMeta(, f138a42e-dee5-4ed6-8256-a2c7957c8703, 4, Finished, Available, Finished)

In [3]:
# --- Control de concurrencia ---
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or ("concurrent" in msg and "delta" in msg)

def run_delta_operation_with_retry(fn, operation_name: str = "operación Delta") -> None:
    """Ejecuta una operación Delta (MERGE, etc.) con reintentos ante ConcurrentAppendException."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            fn()
            if attempt > 1:
                print(f"   {operation_name} completada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada en {operation_name} (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

# --- Función de Logging ---
def update_task_status(status, message, new_watermark=None):
    """Actualiza la tabla de control con el estado final y el nuevo watermark (con reintentos ante concurrencia)."""
    safe_message = message.replace("'", "''")

    watermark_update_sql = ""
    if new_watermark is not None:
        watermark_str = new_watermark.strftime('%Y-%m-%d %H:%M:%S.%f') if isinstance(new_watermark, datetime) else str(new_watermark)
        watermark_update_sql = f", last_watermark_value = '{watermark_str}'"

    update_query = f"""
        UPDATE {control_table_full_name}
        SET
            last_run_status = '{status}',
            last_message = '{safe_message}',
            last_run_at = CAST('{current_utc_timestamp}' AS TIMESTAMP),
            updated_at = CAST('{current_utc_timestamp}' AS TIMESTAMP)
            {watermark_update_sql}
        WHERE
            task_id = {task_id}
    """

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(update_query)
            print(f"Log updated: Status='{status}', Message='{message}'" + (f" (intento {attempt})" if attempt > 1 else ""))
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada al actualizar control (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(f"FATAL: Could not update control table log. Reason: {e}")
                raise
    if last_error is not None:
        raise last_error

def apply_company_code_fix(
    df,
    col_name="company_code",
    mapping=None,
    normalize=True,
    coerce_non_string=False,
    verbose=True
):
    """
    Normaliza company_code:
      ST -> STU
      PS -> PRO
    - Idempotente: si no existe la columna, no hace nada.
    - Usa map lookup con sintaxis moderna: repl[key_expr]
    """
    # localizar la columna real (case-insensitive)
    real_col = next((c for c in df.columns if c.lower() == col_name.lower()), None)
    if real_col is None:
        if verbose:
            print(f"[apply_company_code_fix] Columna '{col_name}' no existe. Sin cambios.")
        return df

    # mapeo base (+ overrides opcionales)
    base_map = {"ST": "STU", "PS": "PRO"}
    if mapping:
        base_map.update({str(k).upper(): str(v) for k, v in mapping.items()})

    # construir el map dinamicamente
    kv = []
    for k, v in base_map.items():
        kv += [F.lit(k), F.lit(v)]
    repl = F.create_map(*kv)

    # tipo de dato de la columna
    dtype = next(f.dataType for f in df.schema.fields if f.name == real_col)
    is_string = isinstance(dtype, T.StringType)

    if is_string:
        key_expr = F.upper(F.trim(F.col(real_col))) if normalize else F.col(real_col)
        new_value = F.coalesce(repl[key_expr], F.col(real_col))   # << uso de índice []
        return df.withColumn(real_col, new_value)

    if coerce_non_string:
        key_expr = (F.upper(F.trim(F.col(real_col).cast("string")))
                    if normalize else F.col(real_col).cast("string"))
        new_value = F.coalesce(repl[key_expr], F.col(real_col).cast("string")).cast(dtype)
        return df.withColumn(real_col, new_value)

    if verbose:
        print(f"[apply_company_code_fix] '{real_col}' es {dtype}. Omitido (coerce_non_string=False).")
    return df

StatementMeta(, f138a42e-dee5-4ed6-8256-a2c7957c8703, 5, Finished, Available, Finished)

In [4]:
# --- Bloque Principal de Ejecución ---
source_df = None
try:
    # 1. LEER METADATOS DE LA TAREA
    print(f"--- Starting Materialization Task ID: {task_id} ---")
    # Asegúrate de que tu tabla de control tenga una columna task_id para identificar la tarea
    config_df = spark.sql(f"SELECT * FROM {control_table_full_name} WHERE task_id = {task_id} AND is_active = true AND load_type = 'Materialization'")

    if config_df.isEmpty(): 
        raise ValueError(f"Materialization Task ID '{task_id}' not found, is disabled, or is not of type 'Materialization'.")
        
    config = config_df.first()
    
    #source_view_full_name = f'{config["source_layer"]}.{config["source_schema"]}.{config["source_table"]}'
    target_table_full_name = f'{config["target_layer"]}.{config["target_schema"]}.{config["target_table"]}'
    primary_keys_str = config["primary_keys"]
    primary_keys_list = [key.strip() for key in primary_keys_str.split(',')]
    watermark_column = config["watermark_column"]
    last_watermark_value = config["last_watermark_value"]
    
    # 2. LEER DATOS INCREMENTALES DE LA VISTA DE ORIGEN
    source_template = config["materialize_sql"]
    watermark_filter_sql = "" 

    if watermark_column and last_watermark_value:
        # Prepara el fragmento de SQL que se insertará
        watermark_filter_sql = f" AND {watermark_column} >= '{last_watermark_value}'"

    # Reemplaza el placeholder en la plantilla con el filtro (o con nada si no hay marca de agua)
    source_query = source_template.replace("{WATERMARK_FILTER}", watermark_filter_sql)        

    #print(source_query)
    
    unclean_source_df = spark.sql(source_query)
    #source_df = spark.sql(source_query)
    
    if unclean_source_df.isEmpty():
        print("No new records to process for materialization.")
        update_task_status('Success', 'No new records to process.')
        notebookutils.notebook.exit("No new records.")

    # ========================================================================
    # === INICIO: Bloque de Deduplicación ===
    # ========================================================================
    print("Aplicando paso de deduplicación para garantizar unicidad en la clave de MERGE...")

    window_spec = Window.partitionBy(*primary_keys_list).orderBy(col(watermark_column).desc())
    source_with_rownum = unclean_source_df.withColumn("row_num", row_number().over(window_spec))
    source_df = source_with_rownum.filter(col("row_num") == 1).drop("row_num")

    print(f"Datos de origen deduplicados. Registros originales: {unclean_source_df.count()}, Registros después: {source_df.count()}")
    # ========================================================================
    # === FIN: Bloque de Deduplicación ===
    # ========================================================================
    
    # 4. Aplica SI y solo si existe company_code
    #source_df = apply_company_code_fix(source_df)
    #display(source_df.limit(100)) 

    #print("llegamos aqui")    

    source_df.cache()

    #print("llegamos aca")
    
    new_watermark = source_df.agg({watermark_column: "max"}).collect()[0][0]
    #rint(f"Incremental read from view '{source_view_full_name}'. Records read: {source_df.count()}. New watermark: {new_watermark}")

    # 3. CREAR TABLA DE DESTINO SI NO EXISTE
    if not spark.catalog.tableExists(target_table_full_name):
        print(f"Target materialized table '{target_table_full_name}' does not exist. Creating it now...")
        source_df.limit(0).write.format("delta").saveAsTable(target_table_full_name)
        print("Materialized table created successfully.")

    #print("llegamos aqui tambien")
    
    # 4. HACER MERGE EN LA TABLA FÍSICA DE DESTINO
    source_df.createOrReplaceTempView("materialization_source_view")

    #print("despues de createOrReplaceTempView")
    
    merge_on_condition = " AND ".join([f"Target.{key} = Source.{key}" for key in primary_keys_list])
    update_set = ", ".join([f"Target.{c} = Source.{c}" for c in source_df.columns if c not in primary_keys_list])
    insert_cols = ", ".join(source_df.columns)
    insert_values = ", ".join([f"Source.{c}" for c in source_df.columns])

    #print("antes del merge")
    
    merge_sql = f"""
        MERGE INTO {target_table_full_name} Target
        USING materialization_source_view Source ON {merge_on_condition}
        WHEN MATCHED THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_values})
    """

    def _do_merge():
        spark.sql(merge_sql)
    run_delta_operation_with_retry(_do_merge, f"MERGE en {target_table_full_name}")
    print(f"MERGE into materialized table {target_table_full_name} completed successfully.")

    # 5. REGISTRAR ÉXITO Y NUEVO WATERMARK
    update_task_status('Success', f'Task completed successfully. Processed {source_df.count()} records.', new_watermark)
    
except Exception as e:
    error_message = str(e).replace('\n', ' ').replace('\r', '')
    print(f"ERROR: An exception occurred: {error_message}")
    update_task_status('Failed', error_message)
finally:
    if source_df:
        source_df.unpersist()
    print("--- Proceso Finalizado ---")

StatementMeta(, f138a42e-dee5-4ed6-8256-a2c7957c8703, 6, Finished, Available, Finished)

--- Starting Materialization Task ID: 51 ---
Aplicando paso de deduplicación para garantizar unicidad en la clave de MERGE...
Datos de origen deduplicados. Registros originales: 2522628, Registros después: 2522628
llegamos aqui
llegamos aca
Target materialized table 'lh_silver_erp.fi.expenses' does not exist. Creating it now...
Materialized table created successfully.
llegamos aqui tambien
despues de createOrReplaceTempView
antes del merge
MERGE into materialized table lh_silver_erp.fi.expenses completed successfully.
Log updated: Status='Success', Message='Task completed successfully. Processed 2522628 records.'
--- Proceso Finalizado ---
